In [ ]:
import sys
import json
from pathlib import Path

# Add the parent directory to sys.path so we can import synth_extract
sys.path.insert(0, str(Path.cwd().parent)) 

from synth_extract.agents import ExtractorAgent

### Understanding Schema

In [ ]:
import importlib
importlib.reload(sys.modules['synth_extract.agents.schemas'])

In [ ]:
from synth_extract.agents.schemas import build_extraction_parser

In [ ]:
parser = build_extraction_parser()

In [ ]:
parser.pydantic_object.model_json_schema()

### Calling the Agent

In [ ]:
agent = ExtractorAgent(model_name="openai/gpt-5-mini", temperature=0.0)

In [ ]:
print(agent.llm_config())

In [ ]:
pdf_library_path = Path("../../pdf_library")
md_paths = []
# Go through the folders in the library and in each folder you get a markdown file.
for folder in pdf_library_path.iterdir():
    if folder.is_dir():
        print(f"Processing folder: {folder.name}")
        # Look for a markdown file in the folder. There should only be one.
        md_files = list(folder.glob("*.md"))
        if len(md_files) == 0:
            print(f"No markdown file found in {folder.name}, skipping.")
            continue
        elif len(md_files) > 1:
            print(f"Multiple markdown files found in {folder.name}, skipping.")
            continue
        md_file = md_files[0]
        print(f"Found markdown file: {md_file.name}")
        # add the path to .md to a list of paths to process
        md_paths.append(md_file)

In [ ]:
md_path = md_paths[4]
md_path

In [ ]:
markdown_text = open(md_path).read()

In [ ]:
res = agent.extract(markdown_text)

In [ ]:
res.paper

In [ ]:
res.samples

In [ ]:
import pandas as pd
from IPython.display import display

# Convert list of polymer entries into a DataFrame
polymers_df = pd.DataFrame([poly.model_dump() if hasattr(poly, 'model_dump') else dict(poly) for poly in res.samples])

# Display the table with wrapped text so long procedure values do not expand the layout
styled = (
    polymers_df.style
    .set_table_styles([
        {
            'selector': 'th',
            'props': [('text-align', 'left'), ('padding', '8px'), ('font-weight', 'bold')]
        },
        {
            'selector': 'td',
            'props': [
                ('white-space', 'pre-wrap'),
                ('word-wrap', 'break-word'),
                ('max-width', '320px'),
                ('padding', '8px'),
                ('text-align', 'left')
            ]
        },
        {
            'selector': 'table',
            'props': [('table-layout', 'fixed'), ('width', '100%')]
        }
    ])
)

display(styled)